In [1]:
import sys, os
from pathlib import Path

os.environ['USE_CUPY'] = 'false'

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline


CWD: /global/home/hpc5656/SLAM


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from src.utils.map import load_obstacles_config
from src.classes.mapping import LidarGridMapVec
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.belief_quantized.belief_mdp_n import BeliefMDP_n_SLAM as BeliefMDP_n

# Output dir
OUTDIR = PROJECT_ROOT / 'notebooks' / 'outputs' / 'tmat_visuals'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Figures will be saved to:', OUTDIR)


ℹ Using scipy.spatial.KDTree (NumPy backend)
Figures will be saved to: /global/home/hpc5656/SLAM/notebooks/outputs/tmat_visuals


In [3]:
def build_mdp(n=5, dt=1.0, max_a=5.0):
    all_obstacles, area = load_obstacles_config(environment='toy2')
    model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=dt, max_a=max_a)
    sensor = LIDAR(fov=360, r_max=10.0, B=8)
    grid_map = LidarGridMapVec(x_min=area[0], x_max=area[1], y_min=area[2], y_max=area[3], quantization_level=n)
    mdp = BeliefMDP_n(n=n, motion_model=model, measurement_model=sensor, obstacles=all_obstacles, _map=grid_map)
    return mdp


def position_marginal(T_col, X_n, n_side):
    pos = X_n[:, :2]
    xs = np.unique(pos[:, 0]); ys = np.unique(pos[:, 1])
    x_to_idx = {v: i for i, v in enumerate(sorted(xs)[:n_side])}
    y_to_idx = {v: i for i, v in enumerate(sorted(ys)[:n_side])}
    heat = np.zeros((n_side, n_side))
    for s, p in enumerate(T_col):
        x, y = pos[s]
        ix = x_to_idx.get(x); iy = y_to_idx.get(y)
        if ix is not None and iy is not None:
            heat[iy, ix] += p
    return heat


def plot_case(mdp: BeliefMDP_n, i_state: int, k_action: int, title: str):
    T = mdp.T_mat
    X = mdp.SQ.X_n
    u = mdp.AQ.U[k_action]
    dt = mdp.motion_model.dt

    # Get full state (4D for DoubleIntegrator)
    x0_full = X[i_state]  # (4,) for DoubleIntegrator: [px, py, vx, vy]
    x0_pos = x0_full[:2]  # Position only for visualization
    
    # Compute deterministic next state using motion model
    x1_full = mdp.motion_model.f_bar(x0_full, u)  # (4,)
    x1_pos = x1_full[:2]  # Next position
    x1_vel = x1_full[2:]  # Next velocity (for info)

    col = T[:, i_state, k_action]
    heat = position_marginal(col, X, n_side=mdp.n)

    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: Position space with initial state and expected displacement
    axs[0].scatter(X[:, 0], X[:, 1], s=5, c='lightgray', alpha=0.3, label='quantized states')
    # Draw arrow from initial to expected next position
    dx = x1_pos[0] - x0_pos[0]
    dy = x1_pos[1] - x0_pos[1]
    axs[0].arrow(x0_pos[0], x0_pos[1], dx, dy, head_width=0.3, head_length=0.2, 
                 fc='red', ec='red', lw=2, length_includes_head=True, label='expected displacement')
    axs[0].plot([x0_pos[0]], [x0_pos[1]], marker='*', color='k', ms=15, label='initial state')
    axs[0].set_title(f"State {i_state}, action {k_action}, |u|={np.linalg.norm(u):.3f}\n"
                     f"Expected: pos→[{x1_pos[0]:.2f}, {x1_pos[1]:.2f}], vel→[{x1_vel[0]:.2f}, {x1_vel[1]:.2f}]")
    axs[0].set_xlabel('x'); axs[0].set_ylabel('y')
    axs[0].set_aspect('equal', adjustable='box')
    axs[0].grid(True, ls=':', alpha=0.5)
    axs[0].legend(fontsize=8)

    # Right: Heatmap of transition probabilities (position-marginalized)
    im = axs[1].imshow(heat, origin='lower', cmap='viridis', aspect='auto')
    fig.colorbar(im, ax=axs[1], fraction=0.046, label='probability')
    axs[1].set_title("T_mat position-marginalized (over velocity)")
    axs[1].set_xlabel('x bin index'); axs[1].set_ylabel('y bin index')
    
    # Mark the initial position bin
    pos = X[:, :2]
    xs = np.unique(pos[:, 0]); ys = np.unique(pos[:, 1])
    x_to_idx = {v: i for i, v in enumerate(sorted(xs)[:mdp.n])}
    y_to_idx = {v: i for i, v in enumerate(sorted(ys)[:mdp.n])}
    ix0 = x_to_idx.get(x0_pos[0])
    iy0 = y_to_idx.get(x0_pos[1])
    if ix0 is not None and iy0 is not None:
        axs[1].plot(ix0, iy0, 'w*', markersize=15, markeredgecolor='black', markeredgewidth=1)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    fig.tight_layout()

    # Display inline
    plt.show()

    # Save to disk as well
    outfile = OUTDIR / f"tmat_visual_{title.replace(' ', '_')}.png"
    fig.savefig(outfile, dpi=150, bbox_inches='tight')
    print('Saved figure to', outfile)


In [8]:
# Build or load MDP (this will compute and cache T_mat on first run)
mdp = build_mdp(n=3, dt=1, max_a=2.0)
print('T_mat shape:', mdp.T_mat.shape)
print('State space size:', mdp.SQ.m_n, 'Action space size:', mdp.AQ.n_u)
print(f'Action space: square-lattice + circle mapping (n={mdp.AQ.n} quantization levels)')
print(f'Action magnitudes: min={np.min(np.linalg.norm(mdp.AQ.U, axis=1)):.3f}, max={np.max(np.linalg.norm(mdp.AQ.U, axis=1)):.3f}')
print()


Computing T_mat for the first time...


Computing T_mat: 100%|██████████| 9/9 [02:22<00:00, 15.79s/it]

Saved T_mat to /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n3_map3x3_max2.0_c2818dab.npz
T_mat shape: (81, 81, 9)
State space size: 81 Action space size: 9
Action space: square-lattice + circle mapping (n=3 quantization levels)
Action magnitudes: min=0.000, max=1.663



In [ ]:
# Standalone action quantizer test (fast, doesn't require MDP construction)
def test_action_quantizer_standalone(n=4, max_acc=2.0):
    """Test ActionQuantizer directly without building the full MDP."""
    from src.classes.quantizer import ActionQuantizer, SquareLatticeQuantizer
    
    print("=" * 70)
    print("STANDALONE ACTION QUANTIZER TEST")
    print("=" * 70)
    print(f"Parameters: n={n}, max_acc={max_acc}")
    print()
    
    # Create action quantizer
    aq = ActionQuantizer(max_acc=max_acc, n=n)
    U = aq.U
    
    # Get square lattice points (before mapping) for comparison
    square_lattice = SquareLatticeQuantizer(
        x_min=-max_acc, x_max=max_acc,
        y_min=-max_acc, y_max=max_acc,
        n=n
    )
    square_points = square_lattice.get_quantized_points()
    
    # Manually apply the mapping to see what happens
    def map_to_circle(x, y):
        """Apply the mapping function f(x,y) = (x*sqrt(1-(y**2)/2), y*sqrt(1-(x**2)/2))"""
        u = x * np.sqrt(np.clip(1 - (y**2) / 2, 0, 1))
        v = y * np.sqrt(np.clip(1 - (x**2) / 2, 0, 1))
        return np.array([u, v])
    
    # Map all square points manually
    mapped_points = np.array([map_to_circle(p[0], p[1]) for p in square_points])
    
    # Compute magnitudes and statistics
    magnitudes = np.linalg.norm(U, axis=1)
    max_mag = np.max(magnitudes)
    min_mag = np.min(magnitudes)
    mapped_mags = np.linalg.norm(mapped_points, axis=1)
    
    # Check for duplicates and mapping issues
    from scipy.spatial.distance import pdist
    
    # Find duplicates in mapped points (before filtering)
    duplicate_pairs = []
    if len(mapped_points) > 1:
        mapped_distances = pdist(mapped_points)
        duplicate_threshold = 1e-6
        for i in range(len(mapped_points)):
            for j in range(i+1, len(mapped_points)):
                if np.linalg.norm(mapped_points[i] - mapped_points[j]) < duplicate_threshold:
                    duplicate_pairs.append((i, j, square_points[i], square_points[j], mapped_points[i]))
    
    # Check which points got filtered
    filtered_indices = []
    for i, mp in enumerate(mapped_points):
        mag = np.linalg.norm(mp)
        if mag > max_acc + 1e-10:
            filtered_indices.append(i)
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Left: Square lattice (before mapping) with annotations
    ax = axes[0]
    scatter1 = ax.scatter(square_points[:, 0], square_points[:, 1], s=50, alpha=0.7, 
                         c=mapped_mags, cmap='viridis', label='Square lattice')
    # Annotate points with indices
    for i, (x, y) in enumerate(square_points):
        ax.annotate(f'{i}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)
    # Draw square boundary
    square_bound = np.array([[-max_acc, -max_acc], [max_acc, -max_acc], 
                             [max_acc, max_acc], [-max_acc, max_acc], [-max_acc, -max_acc]])
    ax.plot(square_bound[:, 0], square_bound[:, 1], 'r--', linewidth=2, label='Square boundary')
    # Draw sqrt(2) circle to show valid domain
    sqrt2_circle = np.linspace(0, 2*np.pi, 100)
    sqrt2_x = np.sqrt(2) * np.cos(sqrt2_circle)
    sqrt2_y = np.sqrt(2) * np.sin(sqrt2_circle)
    ax.plot(sqrt2_x, sqrt2_y, 'g:', linewidth=1.5, label='sqrt(2) boundary (valid domain)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Square Lattice (before mapping)\nn={n}, points={len(square_points)}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
    plt.colorbar(scatter1, ax=ax, label='Mapped magnitude')
    
    # Right: Mapped circle (after mapping) with annotations
    ax = axes[1]
    # Color code by magnitude
    scatter2 = ax.scatter(U[:, 0], U[:, 1], c=magnitudes, s=80, alpha=0.8, cmap='viridis', 
                         edgecolors='black', linewidths=0.5)
    # Annotate points - try to match with square points
    for i, (ux, uy) in enumerate(U):
        # Find closest mapped point
        dists = np.linalg.norm(mapped_points - np.array([ux, uy]), axis=1)
        closest_idx = np.argmin(dists)
        if dists[closest_idx] < 1e-6:
            ax.annotate(f'{closest_idx}', (ux, uy), xytext=(5, 5), textcoords='offset points', 
                       fontsize=8, color='red', weight='bold')
        else:
            ax.annotate(f'?', (ux, uy), xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Draw circle boundary
    theta_circle = np.linspace(0, 2*np.pi, 100)
    circle_x = max_acc * np.cos(theta_circle)
    circle_y = max_acc * np.sin(theta_circle)
    ax.plot(circle_x, circle_y, 'r--', linewidth=2, label=f'Circle boundary (r={max_acc:.2f})')
    
    plt.colorbar(scatter2, ax=ax, label='Magnitude ||u||')
    
    ax.set_xlabel('u_x')
    ax.set_ylabel('u_y')
    ax.set_title(f'Action Space (after circle mapping)\nn={n}, points={len(U)}, '
                 f'max_mag={max_mag:.3f}, min_mag={min_mag:.3f}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
    
    fig.tight_layout()
    plt.show()
    
    # Print detailed statistics
    print(f"Quantization parameter n: {n}")
    print(f"Max acceleration: {max_acc:.3f}")
    print(f"Square lattice points: {len(square_points)}")
    print(f"Mapped points (before filtering): {len(mapped_points)}")
    print(f"Final action points: {len(U)}")
    print(f"Expected points (n²): {n*n}")
    print(f"Points filtered out: {len(filtered_indices)}")
    if filtered_indices:
        print(f"  Filtered indices: {filtered_indices}")
        print(f"  Filtered square points:")
        for idx in filtered_indices:
            sq_pt = square_points[idx]
            mp_pt = mapped_points[idx]
            mp_mag = np.linalg.norm(mp_pt)
            print(f"    [{idx}] square={sq_pt}, mapped={mp_pt}, ||mapped||={mp_mag:.4f}")
    
    print(f"\nMagnitude statistics:")
    print(f"  Min magnitude: {min_mag:.6f}")
    print(f"  Max magnitude: {max_mag:.6f}")
    print(f"  All points within circle (||u|| <= {max_acc:.3f}): {np.all(magnitudes <= max_acc + 1e-10)}")
    print(f"  Points exceeding circle: {np.sum(magnitudes > max_acc + 1e-10)}")
    
    # Check for duplicate points
    print(f"\nDuplicate detection:")
    if len(mapped_points) > 1:
        mapped_distances = pdist(mapped_points)
        min_mapped_dist = np.min(mapped_distances)
        duplicate_threshold = 1e-6
        num_duplicates = np.sum(mapped_distances < duplicate_threshold)
        print(f"  Mapped points (before filtering):")
        print(f"    Minimum distance: {min_mapped_dist:.8f}")
        print(f"    Duplicate pairs (dist < {duplicate_threshold}): {num_duplicates}")
        if duplicate_pairs:
            print(f"    Duplicate pairs found:")
            for i, j, sq1, sq2, mp in duplicate_pairs:
                print(f"      [{i}] square={sq1} -> mapped={mp}")
                print(f"      [{j}] square={sq2} -> mapped={mp}")
                print(f"      Distance: {np.linalg.norm(sq1 - sq2):.6f}")
    
    if len(U) > 1:
        final_distances = pdist(U)
        min_final_dist = np.min(final_distances)
        duplicate_threshold = 1e-10
        num_duplicates_final = np.sum(final_distances < duplicate_threshold)
        print(f"  Final points (after filtering):")
        print(f"    Minimum distance: {min_final_dist:.8f}")
        print(f"    Duplicate points (dist < {duplicate_threshold}): {num_duplicates_final}")
    
    # Show mapping table for points near boundaries
    print(f"\nMapping details (points with |x| or |y| > sqrt(2) ≈ 1.414):")
    sqrt2 = np.sqrt(2)
    boundary_points = []
    for i, (x, y) in enumerate(square_points):
        if abs(x) > sqrt2 - 0.1 or abs(y) > sqrt2 - 0.1:
            mp = mapped_points[i]
            mp_mag = np.linalg.norm(mp)
            arg1 = 1 - (y**2) / 2
            arg2 = 1 - (x**2) / 2
            boundary_points.append((i, x, y, mp[0], mp[1], mp_mag, arg1, arg2))
    
    if boundary_points:
        print(f"  {'Idx':<4} {'Square x':<8} {'Square y':<8} {'Mapped u':<8} {'Mapped v':<8} {'||mapped||':<10} {'arg1':<8} {'arg2':<8}")
        print(f"  {'-'*70}")
        for i, x, y, u, v, mag, a1, a2 in sorted(boundary_points, key=lambda p: abs(p[1]) + abs(p[2])):
            print(f"  {i:<4} {x:>8.3f} {y:>8.3f} {u:>8.4f} {v:>8.4f} {mag:>10.4f} {a1:>8.3f} {a2:>8.3f}")
    else:
        print(f"  No points near sqrt(2) boundary")
    
    # Show all mappings for small n
    if n <= 6:
        print(f"\nComplete mapping table (all {len(square_points)} points):")
        print(f"  {'Idx':<4} {'Square x':<8} {'Square y':<8} {'Mapped u':<8} {'Mapped v':<8} {'||mapped||':<10} {'In Final':<8}")
        print(f"  {'-'*75}")
        for i, (x, y) in enumerate(square_points):
            mp = mapped_points[i]
            mp_mag = np.linalg.norm(mp)
            # Check if this point is in final U
            dists_to_U = np.linalg.norm(U - mp, axis=1)
            in_final = "Yes" if np.min(dists_to_U) < 1e-6 else "No"
            print(f"  {i:<4} {x:>8.3f} {y:>8.3f} {mp[0]:>8.4f} {mp[1]:>8.4f} {mp_mag:>10.4f} {in_final:>8}")
    
    # Save figure
    outfile = OUTDIR / "action_space_quantization_standalone.png"
    fig.savefig(outfile, dpi=150, bbox_inches='tight')
    print(f"\nSaved figure to {outfile}")
    print(f"=" * 70)
    
    return aq


In [ ]:

# ============================================================================
# STANDALONE ACTION QUANTIZER TEST
# ============================================================================
# This test is FAST and INDEPENDENT - it doesn't require building the MDP!
# Use this to validate the quantizer works correctly before spending time
# on the expensive T_mat computation.
# ============================================================================

test_action_quantizer_standalone(n=4, max_acc=2.0)


In [ ]:
# Choose central state and sort actions by magnitude
i_state = mdp.SQ.m_n // 2
mags = np.linalg.norm(mdp.AQ.U, axis=1)
k_sorted = np.argsort(mags)

# Case 1: infeasible pair (if exists)
k_infeasible = None
for k in range(mdp.AQ.n_u):
    if not mdp.K_mask[i_state, k]:
        k_infeasible = k
        break
if k_infeasible is not None:
    plot_case(mdp, i_state, k_infeasible, title='infeasible_pair')
    assert np.allclose(mdp.T_mat[:, i_state, k_infeasible], 0.0)

# Case 2: small feasible control (stay in same pos bin)
# Find the smallest feasible action (if any exist)
k_small = None
for k in k_sorted:
    if mdp.K_mask[i_state, k]:
        k_small = k
        break
if k_small is not None:
    print(f"Small feasible control: action {k_small}, magnitude={np.linalg.norm(mdp.AQ.U[k_small]):.3f}")
    plot_case(mdp, i_state, k_small, title='small_control')
else:
    print(f"Warning: No small feasible action found for state {i_state}, skipping small control case")

# Case 3: largest feasible control
# Always find the largest feasible action (among all feasible ones)
feasible_actions = [k for k in range(mdp.AQ.n_u) if mdp.K_mask[i_state, k]]
if len(feasible_actions) > 0:
    # Find the one with largest magnitude among feasible ones
    feasible_mags = [np.linalg.norm(mdp.AQ.U[k]) for k in feasible_actions]
    k_largest = feasible_actions[np.argmax(feasible_mags)]
    print(f"Largest feasible control: action {k_largest}, magnitude={np.linalg.norm(mdp.AQ.U[k_largest]):.3f}")
    plot_case(mdp, i_state, k_largest, title='largest_control')
else:
    print(f"Warning: No feasible actions found for state {i_state}, skipping largest control case")
    print(f"State {i_state}: {mdp.SQ.X_n[i_state]}")
    print(f"Trying a different state...")
    # Try a few other states to find one with feasible actions
    for alt_state in [0, mdp.SQ.m_n // 4, 3 * mdp.SQ.m_n // 4]:
        alt_feasible = [k for k in range(mdp.AQ.n_u) if mdp.K_mask[alt_state, k]]
        if len(alt_feasible) > 0:
            print(f"Found feasible actions for state {alt_state}, using that instead")
            feasible_mags = [np.linalg.norm(mdp.AQ.U[k]) for k in alt_feasible]
            k_largest = alt_feasible[np.argmax(feasible_mags)]
            plot_case(mdp, alt_state, k_largest, title='largest_control')
            break
